In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

In [2]:
#concatenating the two datasets

df = pd.concat([pd.read_csv(r"../data/student-mat.csv", sep=";"), pd.read_csv(r"../data/student-por.csv", sep=";")], ignore_index=True)
df = df.copy()


In [3]:
#column dropping
df.drop(columns = ['higher', 
                   'Mjob', 'Fjob', 'reason',
                     'paid'], inplace = True)

In [4]:
#more dropping

df.drop(columns=["school", "Medu", "Fedu", "Dalc", "Walc"], inplace=True) 

df["label"] = (df["G3"] >= 10).astype(int) #binary level classification; if g3>=10 then 1(pass) else 0(fail)

df.drop(columns=["G1","G2","G3"], inplace=True) #only label column is g3, dropping g1 and g2

In [5]:
#getting categorical and numerical columns. Numerical columns will be scaled and categorical columns will be encoded.

cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = df.select_dtypes(include=["int64","float64"]).columns.difference(["label"]).tolist()



C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10272\176413831.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [6]:
#splitting data for models
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["label"]), df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]

)



#preprocessing categorical and numerical features for all models

preproc = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])



In [7]:
#Pipeline for Random Forest Classifier
pipe_rf = ImbPipeline([
    ("pre", preproc),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1)),
])
param_dist_rf = {
    "clf__n_estimators": [100, 200, 500],
    "clf__max_depth": [None, 8, 12, 20],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", "log2", 0.5],
    "clf__class_weight": ["balanced", "balanced_subsample", None]
}
search_rf = GridSearchCV(pipe_rf, param_grid=param_dist_rf, scoring="roc_auc", cv=StratifiedKFold(5), n_jobs=-1, verbose=1, refit=True)
search_rf.fit(X_train, y_train)
rf_model = search_rf.best_estimator_  # This is the tuned RF pipeline

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


In [8]:
# Pipeline for Gradient Boosting Classifier
pipe_gb = ImbPipeline([
    ("pre", preproc),
    ("smote", SMOTE(random_state=42)),
    ("clf", GradientBoostingClassifier(random_state=42)),
])
param_dist_gb = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [3, 5, 7],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__subsample": [0.7, 0.8, 0.9],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
}
search_gb = GridSearchCV(pipe_gb, param_grid=param_dist_gb, scoring="roc_auc", cv=StratifiedKFold(5), n_jobs=-1, verbose=1, refit=True)
search_gb.fit(X_train, y_train)
gb_model = search_gb.best_estimator_  # This is the tuned GB pipeline

Fitting 5 folds for each of 729 candidates, totalling 3645 fits


In [9]:
#Pipeline for Support Vector Machine
pipe_svm = ImbPipeline([
    ("pre", preproc),
    ("smote", SMOTE(random_state=42)),
    ("clf", SVC(random_state=42, probability=True)),
])
param_dist_svm = {
    "clf__C": [0.1, 1, 10],
    "clf__kernel": ["linear", "rbf"],
    "clf__gamma": ["scale", "auto"],
}
search_svm = GridSearchCV(pipe_svm, param_grid=param_dist_svm, scoring="roc_auc", cv=StratifiedKFold(5), n_jobs=-1, verbose=1, refit=True)
search_svm.fit(X_train, y_train)
svm_model = search_svm.best_estimator_  # This is the tuned SVM pipeline

Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [10]:
#ensembling models
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[('rf', rf_model), ('gb', gb_model), ('svm', svm_model)],
    voting='soft'  # Averages probabilities
)
ensemble.fit(X_train, y_train)  # This combines the trained models without retraining them

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('rf', ...), ('gb', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use

In [11]:
y_pred_ensemble = ensemble.predict(X_test)
print(classification_report(y_test, y_pred_ensemble))
print("ROC AUC:", roc_auc_score(y_test, ensemble.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0       0.50      0.30      0.38        46
           1       0.82      0.91      0.87       163

    accuracy                           0.78       209
   macro avg       0.66      0.61      0.62       209
weighted avg       0.75      0.78      0.76       209

ROC AUC: 0.6452387303280874


In [12]:
df["label"].value_counts()

label
1    814
0    230
Name: count, dtype: int64

In [13]:
data_to_predict = pd.DataFrame({
    "sex": ["F"],
    "age": [18],
    "address": ["U"],
    "famsize": ["GT3"],
    "Pstatus": ["A"],
    "Mjob": ["at_home"],
    "Fjob": ["teacher"],
    "reason": ["course"],
    "guardian": ["mother"],
    "traveltime": [2],
    "studytime": [2],
    "failures": [0],
    "schoolsup": ["yes"],
    "famsup": ["no"],
    "paid": ["no"],
    "activities": ["no"],
    "nursery": ["yes"],
    "higher": ["yes"],
    "internet": ["no"],
    "romantic": ["no"],
    "famrel": [4],
    "freetime": [3],
    "goout": [4],
    "health": [3],
    "absences": [6],
    "label": [0],
})
ensemble.predict(data_to_predict)

array([1])

In [14]:
with open("stud_performance_classifier.joblib", "wb") as f:
    joblib.dump(ensemble, f)